# LoRA ViT-B/16 — Train, Compare & Bayesian Laplace

Replicates [rezaakb/peft-vit](https://github.com/rezaakb/peft-vit) with Food-101 fine-tuning.
Adds Bayesian uncertainty via Laplace approximation over LoRA weights.

**Runtime → Change runtime type → T4 GPU before running.**

| Cell | What it does |
|------|--------------|
| 1 | Install deps → restart runtime |
| 2 | Clone repos + copy scripts |
| 3 | Mount Drive + configure runs |
| 4 | Train all LoRA configs |
| 5 | Evaluate MAP: Food-101 + Imagenette |
| 6 | Bayesian Laplace over LoRA weights |
| 7 | Plot training curves |

## 1. Install dependencies
*Run once, then Runtime → Restart runtime.*

In [ ]:
!pip install -q --force-reinstall \
    'jsonargparse[signatures]==4.23.1' \
    'peft>=0.4.0' \
    'transformers>=4.30.0' \
    'torchmetrics>=1.0.0' \
    'laplace-torch'
print('Done — Restart runtime, then continue from cell 2.')

## 2. Clone repos + copy scripts

In [ ]:
import os

if not os.path.exists('/content/peft-vit'):
    !git clone https://github.com/rezaakb/peft-vit /content/peft-vit

if not os.path.exists('/content/lora-vit-food101'):
    !git clone https://github.com/Mtlukasik/lora-vit-food101 /content/lora-vit-food101

!cp /content/lora-vit-food101/train_lora.py /content/train_lora.py
!cp /content/lora-vit-food101/laplace_lora_eval.py /content/laplace_lora_eval.py

print('Ready.')
!ls /content/

## 3. Mount Drive + configure runs
*Edit `RUNS` to change what gets trained.*

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

BASE_DIR = '/content/drive/MyDrive/vit_lora_food101'

def get_save_dir(run_cfg, stage='finetuned', dataset='food101'):
    model_name = 'vitb16-in21k'
    mods = '-'.join(run_cfg['target_modules'])
    run_id = f"r{run_cfg['rank']}_ep{run_cfg['epochs']}_{mods}"
    folder = f'pretrained_{model_name}' if stage == 'pretrained' else f'finetuned_{model_name}_{dataset}'
    return f'{BASE_DIR}/deterministic/{folder}/{run_id}'

RUNS = [
    {'rank': 4,  'epochs': 20, 'lr': 0.01, 'batch_size': 64, 'target_modules': ['query', 'value']},
    {'rank': 8,  'epochs': 20, 'lr': 0.01, 'batch_size': 64, 'target_modules': ['query', 'value']},
    {'rank': 16, 'epochs': 20, 'lr': 0.01, 'batch_size': 64, 'target_modules': ['query', 'value', 'key']},
]

for run in RUNS:
    run['save_dir'] = get_save_dir(run)
    print(run['save_dir'])

## 4. Train all LoRA configs
*Skip if already trained — checkpoints live on Drive.*

In [ ]:
import os
from datetime import datetime

for i, run_cfg in enumerate(RUNS):
    mods     = ' '.join(run_cfg['target_modules'])
    save_dir = run_cfg['save_dir']
    r, ep, lr, bs = run_cfg['rank'], run_cfg['epochs'], run_cfg['lr'], run_cfg['batch_size']

    # Skip if already trained
    if os.path.exists(os.path.join(save_dir, 'best.pt')):
        print(f'Skipping run {i+1} (best.pt already exists): {save_dir}')
        continue

    print(f'\n{"="*60}')
    print(f'RUN {i+1}/{len(RUNS)} | r={r} epochs={ep} modules={mods}')
    print(f'{"="*60}')

    !PYTHONPATH=/content/peft-vit/src:/content/peft-vit \
        python /content/train_lora.py \
        --rank {r} --epochs {ep} --lr {lr} --batch_size {bs} \
        --save_dir {save_dir} --target_modules {mods}

print('\nAll runs complete.')

## 5. Evaluate MAP models — Food-101 + Imagenette

In [ ]:
import json, glob, os
import torch, torchvision
import torchvision.transforms as T
from torch.utils.data import DataLoader
from transformers import ViTForImageClassification
from peft import LoraConfig, get_peft_model

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
val_tf = T.Compose([
    T.Resize(256), T.CenterCrop(224),
    T.ToTensor(), T.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225])
])

# Food-101 test
food_ds = torchvision.datasets.Food101('/content/data', split='test', transform=val_tf, download=True)
food_dl = DataLoader(food_ds, batch_size=128, shuffle=False, num_workers=2, pin_memory=True)
print(f'Food-101 test: {len(food_ds)} images')

# Imagenette val (pretrained-domain proxy, ~100MB, no auth)
IMAGENET_DIR = f'{BASE_DIR}/imagenette_val'
if not os.path.exists(IMAGENET_DIR):
    print('Downloading Imagenette (~100MB)...')
    !wget -q https://s3.amazonaws.com/fast-ai-imageclas/imagenette2-320.tgz -P /content/
    !tar -xzf /content/imagenette2-320.tgz -C /content/
    import shutil
    shutil.copytree('/content/imagenette2-320/val', IMAGENET_DIR)
    print(f'Saved to {IMAGENET_DIR}')
else:
    print(f'Imagenette val found at {IMAGENET_DIR}')
imagenet_ds = torchvision.datasets.ImageFolder(IMAGENET_DIR, transform=val_tf)
imagenet_dl = DataLoader(imagenet_ds, batch_size=128, shuffle=False, num_workers=2, pin_memory=True)
print(f'Imagenette val: {len(imagenet_ds)} images, {len(imagenet_ds.classes)} classes')

def evaluate(model, dl, label=''):
    model.eval()
    correct, n = 0, 0
    with torch.no_grad():
        for imgs, labels in dl:
            imgs, labels = imgs.to(DEVICE), labels.to(DEVICE)
            with torch.amp.autocast('cuda'):
                logits = model(imgs).logits
            correct += (logits.argmax(1) == labels).sum().item()
            n += len(labels)
    acc = 100 * correct / n
    print(f'  {label:<45} {acc:.2f}%')
    return acc

def load_finetuned(meta):
    run_dir = os.path.dirname(meta['_path'])
    base = ViTForImageClassification.from_pretrained(
        meta['model_name'], num_labels=meta['num_classes'], ignore_mismatched_sizes=True
    )
    cfg = LoraConfig(r=meta['rank'], lora_alpha=meta['lora_alpha'],
                     target_modules=meta['target_modules'], lora_dropout=0.1, bias='none')
    model = get_peft_model(base, cfg)
    model.load_state_dict(torch.load(os.path.join(run_dir, 'best.pt'), map_location=DEVICE))
    return model.to(DEVICE)

def load_imagenet_head(meta):
    run_dir = os.path.dirname(meta['_path'])
    base = ViTForImageClassification.from_pretrained(
        meta['model_name'], num_labels=meta['num_classes'], ignore_mismatched_sizes=True
    )
    cfg = LoraConfig(r=meta['rank'], lora_alpha=meta['lora_alpha'],
                     target_modules=meta['target_modules'], lora_dropout=0.1, bias='none')
    model = get_peft_model(base, cfg)
    model.load_state_dict(torch.load(os.path.join(run_dir, 'best.pt'), map_location=DEVICE))
    pretrained_in1k = ViTForImageClassification.from_pretrained('google/vit-base-patch16-224')
    model.base_model.model.classifier = pretrained_in1k.classifier
    return model.to(DEVICE)

meta_files = sorted(glob.glob(f'{BASE_DIR}/**/meta.json', recursive=True))
results = []
for path in meta_files:
    with open(path) as f:
        meta = json.load(f)
    meta['_path'] = path
    print(f"\n{'─'*60}\nRun: {meta['run_name']}")

    model = load_finetuned(meta)
    food_acc = evaluate(model, food_dl, 'Food-101 test (finetuned head)')
    del model; torch.cuda.empty_cache()

    model = load_imagenet_head(meta)
    in1k_acc = evaluate(model, imagenet_dl, 'Imagenette val (pretrained head)')
    del model; torch.cuda.empty_cache()

    results.append({'run': meta['run_name'], 'rank': meta['rank'],
                    'modules': '+'.join(meta['target_modules']),
                    'food101': food_acc, 'in1k': in1k_acc,
                    'mean': (food_acc + in1k_acc) / 2})

results.sort(key=lambda x: x['mean'], reverse=True)
print(f"\n{'='*72}")
print(f"{'RUN':<35} {'r':>4} {'Food-101':>10} {'Imagenette':>12} {'MEAN':>8}")
print(f"{'─'*72}")
for r in results:
    print(f"{r['run']:<35} {r['rank']:>4} {r['food101']:>9.2f}% {r['in1k']:>11.2f}% {r['mean']:>7.2f}%")
print(f"{'='*72}")
print('\nPaper reference (CIFAR-100 + IN-1k, different dataset — not directly comparable):')
print('  r=4  : CIFAR-100=87.91%  IN-1k=66.82%  MEAN=77.37%')
print('  r=8  : CIFAR-100=88.27%  IN-1k=65.99%  MEAN=77.13%')
print('  r=16 : CIFAR-100=87.84%  IN-1k=65.06%  MEAN=76.45%')

## 6. Bayesian Laplace approximation over LoRA weights

Fits a diagonal Laplace posterior over the LoRA A/B matrices (all other weights stay frozen).
Reports accuracy, NLL, and ECE for MAP vs Laplace — Laplace should improve calibration without hurting accuracy.
Saves `laplace_results.json` and `laplace.pkl` per run.

In [ ]:
import glob, json, os

meta_files = sorted(glob.glob(f'{BASE_DIR}/**/meta.json', recursive=True))
print(f'Found {len(meta_files)} run(s) to process\n')

for meta_path in meta_files:
    la_results_path = os.path.join(os.path.dirname(meta_path), 'laplace_results.json')
    if os.path.exists(la_results_path):
        print(f'Skipping (already done): {meta_path}')
        continue
    print(f'\n{"="*60}')
    print(f'Fitting Laplace: {meta_path}')
    print(f'{"="*60}')
    !PYTHONPATH=/content/peft-vit/src:/content/peft-vit \
        python /content/laplace_lora_eval.py \
        --meta_path {meta_path} \
        --fit_samples 2000

# Summary table
la_files = sorted(glob.glob(f'{BASE_DIR}/**/laplace_results.json', recursive=True))
if la_files:
    print(f"\n{'='*80}")
    print(f"{'RUN':<35} {'Acc':>8} {'MAP NLL':>10} {'LA NLL':>10} {'MAP ECE':>10} {'LA ECE':>10}")
    print(f"{'─'*80}")
    for path in la_files:
        with open(path) as f:
            r = json.load(f)
        print(f"{r['run_name']:<35} "
              f"{r['laplace']['accuracy']:>7.2f}% "
              f"{r['map']['nll']:>10.4f} "
              f"{r['laplace']['nll']:>10.4f} "
              f"{r['map']['ece']:>10.4f} "
              f"{r['laplace']['ece']:>10.4f}")
    print(f"{'='*80}")
    print('NLL and ECE: lower is better. Laplace should improve both without changing accuracy.')

## 7. Plot training curves

In [ ]:
import json, glob
import matplotlib.pyplot as plt

meta_files = sorted(glob.glob(f'{BASE_DIR}/**/meta.json', recursive=True))
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

for path in meta_files:
    with open(path) as f:
        meta = json.load(f)
    label  = f"r={meta['rank']} {'+'.join(meta['target_modules'])}"
    epochs = range(1, len(meta['history']['train_loss']) + 1)
    ax1.plot(epochs, meta['history']['train_loss'], marker='o', markersize=3, label=label)
    ax2.plot(epochs, meta['history']['val_acc'],    marker='o', markersize=3, label=label)

ax1.set_title('Train Loss');            ax1.set_xlabel('Epoch'); ax1.set_ylabel('Loss');  ax1.legend()
ax2.set_title('Val Accuracy (Food-101)'); ax2.set_xlabel('Epoch'); ax2.set_ylabel('Acc %'); ax2.legend()
plt.tight_layout()
out = f'{BASE_DIR}/training_curves.png'
plt.savefig(out, dpi=150)
plt.show()
print(f'Saved to {out}')

## 8. Backup to Google Drive
*Drive is already the save target — this cell is optional for local copies.*

In [ ]:
# All checkpoints and results are already saved to Drive via BASE_DIR.
# To copy Laplace pickles explicitly:
# import glob, shutil
# for p in glob.glob(f'{BASE_DIR}/**/laplace.pkl', recursive=True):
#     print('Already on Drive:', p)